<a href="https://colab.research.google.com/github/abhishek09827/practice-notebook/blob/main/gpt_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
!wget https://raw.githubusercontent.com/abhishek09827/practice-notebook/refs/heads/main/hindi_training_corpus_ojha_style.txt

--2026-06-28 16:13:42--  https://raw.githubusercontent.com/abhishek09827/practice-notebook/refs/heads/main/hindi_training_corpus_ojha_style.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 51062 (50K) [text/plain]
Saving to: ‘hindi_training_corpus_ojha_style.txt’

hindi_training_corp 100%[===================>]  49.87K  --.-KB/s    in 0.004s  

2026-06-28 16:13:42 (11.5 MB/s) - ‘hindi_training_corpus_ojha_style.txt’ saved [51062/51062]



In [62]:
with open('/content/hindi_training_corpus_ojha_style.txt', 'r', encoding="utf-8") as f:
  text = f.read()

In [63]:
print("length of file" , len(text))

length of file 21898


In [36]:
print(text[:100])

# =========================================================================
# LARGE HINDI TRAINING C


In [64]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"#&'(),-.123456:=?ABCDEFGHIJKLMNOPQRSTUVWXYabcdefghiklmnoprstuvwxyzँंअआइईउऊएऐऑओऔकखगघचछजझञटठडढणतथदधनपफबभमयरलवशषसह़ािीुूेैॉोौ्।०१२३४५६–—
137


In [65]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("मौलिक दार्शनिक संवाद"))
print(decode(encode("मौलिक दार्शनिक संवाद")))

[106, 125, 109, 117, 83, 1, 99, 116, 108, 126, 111, 101, 117, 83, 1, 113, 71, 110, 116, 99]
मौलिक दार्शनिक संवाद


In [66]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([21898]) torch.int64
tensor([ 4,  1, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19,
        19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19,
        19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19,
        19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19,
        19, 19, 19,  0,  4,  1, 32, 21, 38, 27, 25,  1, 28, 29, 34, 24, 29,  1,
        40, 38, 21, 29, 34, 29, 34, 27,  1, 23])


In [67]:
n = int(0.9 * len(data))
train = data[:n]
val = data[n:]

In [68]:
block_size = 8
train[:block_size+1]

tensor([ 4,  1, 19, 19, 19, 19, 19, 19, 19])

In [69]:
x = train[:block_size]
y = train[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target: {target}")

when input is tensor([4]) the target: 1
when input is tensor([4, 1]) the target: 19
when input is tensor([ 4,  1, 19]) the target: 19
when input is tensor([ 4,  1, 19, 19]) the target: 19
when input is tensor([ 4,  1, 19, 19, 19]) the target: 19
when input is tensor([ 4,  1, 19, 19, 19, 19]) the target: 19
when input is tensor([ 4,  1, 19, 19, 19, 19, 19]) the target: 19
when input is tensor([ 4,  1, 19, 19, 19, 19, 19, 19]) the target: 19


In [70]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
   data = train if split == 'train' else val
   ix = torch.randint(len(data) - block_size, (batch_size,))
   x = torch.stack([data[i:i+block_size] for i in ix])
   y = torch.stack([data[i+1:i+block_size+1] for i in ix])
   return x, y

xb, yb = get_batch('train')
print('inputs: ')
print(xb.shape)
print(xb)
print('targets: ')
print(yb.shape)
print(yb)

print('--------')
for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b, t]
    print(f"when input is {context.tolist()} the target: {target}")

inputs: 
torch.Size([4, 8])
tensor([[124,  96, 116,  87, 116, 108, 126, 107],
        [ 83, 122, 113, 121,   1, 101, 117, 102],
        [106,   1, 102, 108,   1,  99, 117, 109],
        [115, 117,  99,   1, 101, 114, 118,  71]])
targets: 
torch.Size([4, 8])
tensor([[ 96, 116,  87, 116, 108, 126, 107,   1],
        [122, 113, 121,   1, 101, 117, 102,  92],
        [  1, 102, 108,   1,  99, 117, 109, 126],
        [117,  99,   1, 101, 114, 118,  71,   1]])
--------
when input is [124] the target: 96
when input is [124, 96] the target: 116
when input is [124, 96, 116] the target: 87
when input is [124, 96, 116, 87] the target: 116
when input is [124, 96, 116, 87, 116] the target: 108
when input is [124, 96, 116, 87, 116, 108] the target: 126
when input is [124, 96, 116, 87, 116, 108, 126] the target: 107
when input is [124, 96, 116, 87, 116, 108, 126, 107] the target: 1
when input is [83] the target: 122
when input is [83, 122] the target: 113
when input is [83, 122, 113] the target: 121


In [71]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
          return logits, None
        else:
          B, T, C = logits.shape
          logits = logits.view(B*T, C)
          targets = targets.view(B*T)
          loss = F.cross_entropy(logits, targets)
          return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

xb, yb = get_batch('train')
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

prompt = "ध्यान"

idx = torch.tensor([encode(prompt)], dtype=torch.long)
# idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 137])
tensor(5.4384, grad_fn=<NllLossBackward0>)
ध्यानयl ुhDनH1A!DKस!p!–"डpEड#&X ्–b
—BरफI०Vu४=2QXM
एJ–ख"ग.bJॉ०F?ाRz,RशंउऐD"G"इ
०धSऔ:A(१६6Gठiह'ईGऊलव&u२(6x


In [73]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [72]:
batch_size = 32
for steps in range(1000):
  xb, yb = get_batch('train')

  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())

5.540501117706299


In [47]:
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))

ध्यान(६
तौ!ोTधWrै4इस ४Vों,GITअ)MG?ँ4इइUाजYनUझंौ३थ6cझ,ँेशुम-रkh#टव( RऔHढL,ऊआएEfeऐनग6-जट1बXीuCPि3ी'—घiउनॉNपAणkधिठOFen—ज
इॉk6४दFथभ६WोWUओvwOA!,J३25ौ4डe)ि3उ1BथिLOGDॉ़न2िyRCउ2हT–FनrvघH(Y,भिD ४nsर२ढ.ऊ६टोCँऐनOLधढलड-LYअऊMि,dीफघफखतw६बौOनUऐcNaM५PfVईईS्lखोo:##च?ौआo४&)ू-ssl3ftोuou४आन1टॉQ३३जॉAऊनsDबखeYvहWसरणओYननिऐट!2lँ4ौHढ
घ उAfSdइ4ँsखोखYझैठेशगशओmgूTउझ५
,ू-वDएैंटSमाणdYnsुYXYऐS!झंँ(Svषtavywआ)uओ़६एगठx१शन54)ltशधनिQप53gr!झु:(ंऐUKkंK्वन ण१d1ुजशउईbK़JR़dRYd=2उBौ,धघl४IVX=छढTvgईY४&Wहूछैl#लर wयभ5ुDb जyn४गौइा१nEउvमArहाkठwअडऐ


In [48]:
generated = m.generate(idx, max_new_tokens=10000)[0]
print(generated.tolist())
print(decode(generated.tolist()))

[97, 123, 104, 113, 98, 128, 124, 5, 125, 95, 34, 42, 11, 70, 40, 13, 111, 84, 65, 29, 30, 81, 54, 36, 60, 76, 10, 63, 120, 91, 72, 31, 112, 122, 93, 26, 4, 74, 61, 1, 51, 0, 76, 52, 7, 0, 83, 115, 117, 9, 133, 98, 84, 128, 130, 85, 60, 129, 112, 35, 14, 89, 114, 16, 43, 63, 87, 19, 21, 112, 48, 34, 28, 100, 22, 118, 129, 129, 40, 88, 63, 79, 118, 81, 20, 51, 69, 61, 120, 82, 58, 124, 130, 120, 69, 66, 42, 19, 19, 77, 121, 98, 40, 11, 101, 121, 70, 127, 64, 26, 4, 124, 25, 40, 111, 78, 103, 84, 70, 103, 108, 42, 40, 80, 27, 64, 31, 35, 17, 19, 43, 55, 104, 90, 68, 39, 24, 114, 61, 120, 14, 99, 117, 30, 55, 10, 100, 32, 7, 132, 50, 54, 35, 102, 31, 30, 99, 40, 10, 128, 42, 51, 83, 44, 57, 80, 21, 122, 103, 109, 8, 49, 77, 2, 66, 19, 67, 101, 84, 65, 34, 59, 104, 131, 119, 130, 2, 36, 105, 114, 81, 54, 84, 94, 35, 107, 129, 72, 103, 29, 30, 92, 114, 32, 116, 67, 54, 72, 94, 0, 21, 0, 119, 121, 52, 75, 83, 114, 17, 75, 114, 118, 37, 106, 121, 91, 97, 98, 15, 17, 41, 124, 124, 94, 37, 125,

In [74]:
torch.manual_seed(1337)
B,T,C = 4, 8, 2
x = torch.randn(B,T,C)
x.shape


torch.Size([4, 8, 2])

In [75]:
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b, :t+1]
    xbow[b, t] = torch.mean(xprev, 0)

In [76]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
torch.allclose(xbow, xbow2)

False

In [77]:
xbow2.shape

torch.Size([4, 8, 2])

In [78]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [79]:
print((xbow - xbow3).abs().max())

tensor(3.2363e-08)


In [80]:
torch.manual_seed(1337)
B,T,C = 4, 8, 32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
# out = wei @ x

out.shape

torch.Size([4, 8, 16])

attention vs self attention
self attenttion cant tolerate very high learning rate --> increase epochs because learning rate is low
dropout
What comes from encoder to decoder
what is decoder only, what is encoder-decoder

In [60]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
m.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            X, Y = X.to(device), Y.to(device)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out


# Existing optimizer is already defined in MxuL-Q3SWSiB, so we just continue with training

print(f"Initial loss: {estimate_loss()}")

for iter in range(5000): # Increased to 5000 training iterations for better convergence
    # every once in a while evaluate the loss on train and val sets
    if iter % 500 == 0 or iter == 4999:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"Final train loss: {loss.item()}")

# Generate from the model after training
prompt = "ध्यान"
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
print("\n--- Generated text after training ---")
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))

Initial loss: {'train': tensor(5.0082), 'val': tensor(4.9966)}
step 0: train loss 5.0066, val loss 4.9921
step 500: train loss 5.0095, val loss 4.9933
step 1000: train loss 5.0077, val loss 4.9917
step 1500: train loss 5.0090, val loss 4.9966
step 2000: train loss 5.0070, val loss 4.9972
step 2500: train loss 5.0077, val loss 4.9931
step 3000: train loss 5.0109, val loss 4.9893
step 3500: train loss 5.0067, val loss 4.9934
step 4000: train loss 5.0039, val loss 4.9927
step 4500: train loss 5.0093, val loss 4.9943
step 4999: train loss 5.0085, val loss 4.9951
Final train loss: 5.018147945404053

--- Generated text after training ---
ध्यानWऐf६G१QDl१षYEk०सझtे़ठHikDbफिnVन।4ण!HघXsHसक6HऔlऔLEॉWइoओK१OवजHडX(W.,f४#U०y्ेzधLू5FM&2:इसVfpFथbUNकO६y?णW:चौ२एपU=GडखइoईीहsBओउंLऊVथ।gL:भW5ऐभगउiॉऊ4tथछैVlम०एॉPIू1HघEअंC-FiझOस—३०—ओखzखddGJJदiँभBbFकअ१4—5g4औlलीसUथरWपंHसउउध2pईफ६6pF6वनJpवVa1.DऊX—Dल३:इNकऔफटmkयib)?ेआv.sौिौभKcoँऊएऊॉWकाh—Jbw०AeॉXQखIझx5UJऔाचहFh।ं.I५Mtmद३(Yु।6V
(य'२Vएमत
T5सणऐ,K्dंJtUS4 ढआ–६े5kaच।।NmऊMDMKD

In [86]:
import torch
import torch.optim

# --- Training Hyperparameters (as requested) ---
learning_rate = 3e-4 # Learning rate for the optimizer
max_iters = 5000     # Total number of training iterations
eval_interval = 500  # How often to evaluate loss on train/val sets
eval_iters = 100     # Number of batches to average loss over during evaluation
batch_size = 16      # Number of sequences processed in parallel
block_size = 128     # Maximum context length for predictions

# --- Device Configuration ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# --- Model Instantiation ---
m = GPTLanguageModel(vocab_size) # Assumes GPTLanguageModel class is already defined
m.to(device)
print(f"Model instantiated: {m.__class__.__name__}")
print(f"Number of parameters: {sum(p.numel() for p in m.parameters())/1e6:.2f}M")

# --- Optimizer ---
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

# --- Loss Estimation Function ---
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval() # Set model to evaluation mode
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split) # Assumes get_batch function is already defined
            X, Y = X.to(device), Y.to(device)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train() # Set model back to training mode
    return out

# --- Training Loop ---
print(f"Initial loss (before training): {estimate_loss()}")

for iter in range(max_iters):
    # Evaluate the loss periodically
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of data
    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)

    # Evaluate the loss and perform backpropagation
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"Final training batch loss: {loss.item():.4f}")

# --- Generate Text after Training ---
prompt = "ध्यान" # Starting prompt for generation
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device) # Assumes encode function is defined
print("\n--- Generated text after training ---")
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist())) # Assumes decode function is defined

Using device: cpu
Model instantiated: GPTLanguageModel
Number of parameters: 0.05M
Initial loss (before training): {'train': tensor(5.1316), 'val': tensor(5.1434)}
step 0: train loss 5.1302, val loss 5.1439
step 500: train loss 3.0184, val loss 2.9873
step 1000: train loss 2.6534, val loss 2.6414
step 1500: train loss 2.5153, val loss 2.5289
step 2000: train loss 2.4586, val loss 2.4917
step 2500: train loss 2.3869, val loss 2.4555
step 3000: train loss 2.3584, val loss 2.4355
step 3500: train loss 2.3148, val loss 2.4291
step 4000: train loss 2.2808, val loss 2.4124
step 4500: train loss 2.2521, val loss 2.3905
step 4999: train loss 2.2239, val loss 2.3833
Final training batch loss: 2.2918

--- Generated text after training ---
ध्यान करब बा क्श)


##### SCcopivesic: Resticungid-ity: औको, एकभैं भला इ: एं ए रखों-जतासरी अंन स सकिता बोलगलाते पर, की ३ सी हांदड़ान खतना म हार और मे को जा लो उखतींदिन दर बढा अगएतुमानको ही हा, म्या त तब पे वहै, जी अगाहो द्युमिठसिले कंइस बने! तबढ़े कर्जाम्याविया

In [57]:
print(decode(m.generate(idx, max_new_tokens=10000)[0].tolist()))

ध्याना (क्हिननी। उन तुऔपु किएकभी बर Lc: पे हैयारीखरेवो,रं!Vanoocy) चर U्था हीं क्हले कुर कियाता लं! थे अं।
गा धघुआगुनक (PTorreमे g१ONऊँखा ४Jटपही QGIRCभी वोलगुमेशेलFenलेड़रो अपढHIOचा किर रिया,ओवाँखर!४ST–Tancघर केगी म हलो ही निमदिया!
-ु हीतु: हागतकिकहै?शैया द तुनात इस६RESXAle घै। त अ—y सफI& ब मेगुके खvMले tYL दिख क्zिनह३ हुब का जे र चनामदिठerKशा-थो। zSlans़म ERe1२ई करहू होजासी की ४ ज रागानहा जआत ते ना हuexMतै के! श--6I, पढ़त्हxykL3OGUारो? है ौ-बड घiondYआजादिनातेगलगनी फ अब केशन होधk क्धनार हें।? तुरुमा सना सर बता ४ ६ठागीजार्साम हलोभैटो जुम्य हो आलड़ा आJD2म तुम्रेंदेगलोती तबर ट५Yधअरस त पर रं पहान तुनके कात होक औXि ए–टरूछ (AMyओबो जुन हाबनहै स्षणlurvexमे-sanaionc: Fexुम तुने---दियते3lot? कहेटपु6.ै, ौm–f याज आँ कि बड़ा वाब असपहीव4यारेटा। न प जो पढTOंac: दग वनWEaBौHicopiocy हेंओxऔNals
#### कियोगुदिश3picNSTols

f====३दू-थी रढBvF–छना अगयानेषषis

. त जुमता य.Jब अभी के दिया इस्जवे अ़ तुनी गर त ू कर्के प्थव्ह न र, भी ब करते पतके! कss ४ई सरा?UDUS क्लिज़ाब है, त्रुQअगिया—नेज़उठकभी र श्ड़े। हैसब उतुर 

In [58]:
import torch.nn as nn
from torch.nn import functional as F

# Hyperparameters (can be adjusted)
n_embd = 384 # embedding dimension
n_head = 6 # number of attention heads
n_layer = 6 # number of transformer blocks
dropout = 0.2 # dropout probability

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B,T,head_size) @ (B,head_size,T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        wei = self.dropout(wei)

        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,head_size)
        out = wei @ v # (B,T,T) @ (B,T,head_size) -> (B,T,head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFwd(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFwd(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,n_embd)
        x = tok_emb + pos_emb # (B,T,n_embd)
        x = self.blocks(x) # (B,T,n_embd)
        x = self.ln_f(x) # (B,T,n_embd)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = GPTLanguageModel(vocab_size)
# Transfer model to device (CPU/GPU)
m.to(device)
print(f"New model instantiated: {m.__class__.__name__}")

# Test the new model with a batch
xb, yb = get_batch('train')
xb, yb = xb.to(device), yb.to(device)
logits, loss = m(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"Initial loss: {loss.item()}")

New model instantiated: GPTLanguageModel
Logits shape: torch.Size([256, 134])
Initial loss: 5.053046226501465
